# 🚀 MovieLens KGNN - GPU Accelerated Evaluation,,**Optimized for Google Colab Pro**,,## 📋 Quick Start,1. Runtime → Change runtime type → **T4 GPU** ,2. Runtime → Run all,3. Upload your 3 CSV files when prompted,,---

## 📂 Step 1: Mount Google Drive

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118,import torch, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns,import torch.nn.functional as F,from sklearn.metrics import *,from sklearn.model_selection import train_test_split,from collections import defaultdict,import warnings, time,from tqdm.auto import tqdm,from google.colab import files,warnings.filterwarnings('ignore'),sns.set_style('whitegrid'),plt.rcParams['figure.dpi'] = 300,device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),print('='*60),print(f'🚀 Device: {device}'),if torch.cuda.is_available():,    print(f'✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB)'),else:,    print('⚠️  NO GPU! Change runtime type!'),print('='*60)

## 📦 Step 2: Install & Import Libraries

In [ ]:
print('Upload: users.csv, movies.csv, ratings.csv'),uploaded = files.upload(),print(f"✅ Uploaded {len(uploaded)} files")

## 📊 Step 3: Load & Index Data from Drive

In [ ]:
print('[1/8] Loading data...'),start = time.time(),users = pd.read_csv('users.csv'),movies = pd.read_csv('movies.csv'),ratings = pd.read_csv('ratings.csv'),print(f'  ✓ {len(users):,} users, {len(movies):,} movies, {len(ratings):,} ratings'),,user_ids = sorted(users['UserID'].unique()),movie_ids = sorted(movies['MovieID'].unique()),user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)},movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)},idx_to_user = {idx: uid for uid, idx in user_to_idx.items()},idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()},n_users, n_movies = len(user_ids), len(movie_ids),print(f'  ⏱ {time.time()-start:.1f}s')

## ✂️ Step 4: Train/Test Split

In [ ]:
print('[2/8] Train/test split...'),start = time.time(),train_ratings, test_ratings = [], [],for user_id in tqdm(ratings['UserID'].unique(), desc='  Split'):,    user_ratings = ratings[ratings['UserID'] == user_id],    if len(user_ratings) >= 4:,        train, test = train_test_split(user_ratings, test_size=0.2, random_state=42),        train_ratings.append(train),        test_ratings.append(test),    else:,        train_ratings.append(user_ratings),train_df = pd.concat(train_ratings, ignore_index=True),test_df = pd.concat(test_ratings, ignore_index=True) if test_ratings else pd.DataFrame(),print(f'  ✓ Train: {len(train_df):,}, Test: {len(test_df):,}'),print(f'  ⏱ {time.time()-start:.1f}s')

## 🎯 Step 5: Build GPU Matrices (GPU Acceleration Starts Here!)

In [ ]:
print('[3/8] Building GPU matrices...'),start = time.time(),,# Rating matrix,user_movie_matrix = torch.zeros((n_users, n_movies), device=device),for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='  Ratings'):,    user_movie_matrix[user_to_idx[row['UserID']], movie_to_idx[row['MovieID']]] = row['Rating'],,# Genre features,all_genres = sorted(set(g for genres in movies['Genres'].dropna() for g in genres.split('|'))),genre_to_idx = {g: i for i, g in enumerate(all_genres)},movie_genre_matrix = torch.zeros((n_movies, len(all_genres)), device=device),for _, m in movies.iterrows():,    if pd.notna(m['Genres']):,        for g in m['Genres'].split('|'):,            movie_genre_matrix[movie_to_idx[m['MovieID']], genre_to_idx[g]] = 1.0,,# Year features,decades = sorted(movies['Year'].dropna().apply(lambda y: (int(y)//10)*10).unique()),decade_to_idx = {d: i for i, d in enumerate(decades)},movie_year_matrix = torch.zeros((n_movies, len(decades)), device=device),for _, m in movies.iterrows():,    if pd.notna(m['Year']):,        d = (int(m['Year'])//10)*10,        if d in decade_to_idx:,            movie_year_matrix[movie_to_idx[m['MovieID']], decade_to_idx[d]] = 1.0,,# User demographics,genders = sorted(users['Gender'].unique()),ages = sorted(users['Age'].unique()),occupations = sorted(users['Occupation'].unique()),gender_to_idx = {g: i for i, g in enumerate(genders)},age_to_idx = {a: i for i, a in enumerate(ages)},occ_to_idx = {o: i for i, o in enumerate(occupations)},user_demo_matrix = torch.zeros((n_users, len(genders)+len(ages)+len(occupations)), device=device),for _, u in users.iterrows():,    idx = user_to_idx[u['UserID']],    user_demo_matrix[idx, gender_to_idx[u['Gender']]] = 1.0,    user_demo_matrix[idx, len(genders)+age_to_idx[u['Age']]] = 1.0,    user_demo_matrix[idx, len(genders)+len(ages)+occ_to_idx[u['Occupation']]] = 1.0,,print(f'  ✓ Matrices on GPU: {user_movie_matrix.shape}, {movie_genre_matrix.shape}'),print(f'  ⏱ {time.time()-start:.1f}s')

## 🧮 Step 6: Compute Similarities (GPU Matrix Multiplication!)

In [ ]:
print('[4/8] Computing similarities on GPU...'),start = time.time(),,movie_features = torch.cat([movie_genre_matrix, movie_year_matrix], dim=1),movie_similarity = torch.mm(F.normalize(movie_features, p=2, dim=1), F.normalize(movie_features, p=2, dim=1).t()),user_similarity = torch.mm(F.normalize(user_demo_matrix, p=2, dim=1), F.normalize(user_demo_matrix, p=2, dim=1).t()),,high_mask = (user_movie_matrix >= 4.0).float(),low_mask = (user_movie_matrix <= 2.0).float(),rated_mask = (user_movie_matrix > 0).float(),,print(f'  ✓ Similarity matrices: {movie_similarity.shape}, {user_similarity.shape}'),print(f'  ⏱ {time.time()-start:.1f}s')

## 🎬 Step 7: Generate Recommendations (Batch GPU Processing!)

In [ ]:
print('[5/8] Generating recommendations...'),start = time.time(),,def recommend_batch(user_indices, k=100):,    batch = len(user_indices),    user_high = high_mask[user_indices],    user_low = low_mask[user_indices],    user_rated = rated_mask[user_indices],    ,    content = torch.mm(user_high, movie_similarity) - 0.3*torch.mm(user_low, movie_similarity),    collab = torch.mm(user_similarity[user_indices], high_mask),    scores = 2.0*content + 1.0*collab,    scores = scores * (1 - user_rated),    ,    vals, idxs = torch.topk(scores, k, dim=1),    return idxs.cpu().numpy(), vals.cpu().numpy(),,test_users = test_df['UserID'].unique(),test_indices = [user_to_idx[uid] for uid in test_users],user_recs = {},batch_size = 256,,for i in tqdm(range(0, len(test_indices), batch_size), desc='  Batches'):,    batch_idx = test_indices[i:i+batch_size],    batch_users = test_users[i:i+batch_size],    top_movies, top_scores = recommend_batch(batch_idx, k=100),    for j, uid in enumerate(batch_users):,        user_recs[uid] = [(idx_to_movie[m], float(top_scores[j][k])) for k, m in enumerate(top_movies[j])],,print(f'  ✓ Generated recs for {len(user_recs):,} users'),print(f'  ⏱ {time.time()-start:.1f}s')

## 📋 Step 8: Prepare Ground Truth

In [ ]:
print('[6/8] Preparing ground truth...'),user_actual_high = defaultdict(set),user_actual_all = defaultdict(dict),for _, row in test_df.iterrows():,    user_actual_all[row['UserID']][row['MovieID']] = float(row['Rating']),    if row['Rating'] >= 4.0:,        user_actual_high[row['UserID']].add(row['MovieID']),print(f"  ✓ {len(user_actual_high):,} users")

## 📊 Step 9: Compute Metrics

In [ ]:
print('[7/8] Computing metrics...'),,def precision_at_k(recs, actual, k):,    if not recs or not actual: return 0.0,    top_k = [m for m, _ in recs[:k]],    return len(set(top_k) & actual) / k,,def ndcg_at_k(recs, actual_dict, k):,    if not recs: return 0.0,    top_k = [m for m, _ in recs[:k]],    rel = [actual_dict.get(m, 0.0) for m in top_k],    if sum(rel) == 0: return 0.0,    dcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(rel)),    ideal = sorted(actual_dict.values(), reverse=True)[:k],    idcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(ideal)),    return dcg / idcg if idcg > 0 else 0.0,,p1, p3, p5, ndcg = [], [], [], [],for uid in tqdm(user_recs.keys(), desc='  Precision'):,    if uid in user_actual_high and len(user_actual_high[uid]) > 0:,        recs = user_recs[uid],        p1.append(precision_at_k(recs, user_actual_high[uid], 1)),        p3.append(precision_at_k(recs, user_actual_high[uid], 3)),        p5.append(precision_at_k(recs, user_actual_high[uid], 5)),    if uid in user_actual_all:,        ndcg.append(ndcg_at_k(user_recs[uid], user_actual_all[uid], 5)),,p_at_1 = np.mean(p1) if p1 else 0,p_at_3 = np.mean(p3) if p3 else 0,p_at_5 = np.mean(p5) if p5 else 0,ndcg_5 = np.mean(ndcg) if ndcg else 0,,y_true, y_score = [], [],for uid in user_recs.keys():,    if uid in user_actual_all:,        for mid, score in user_recs[uid][:50]:,            if mid in user_actual_all[uid]:,                y_true.append(1 if user_actual_all[uid][mid] >= 4.0 else 0),                y_score.append(score),,if len(set(y_true)) > 1:,    auc_roc = roc_auc_score(y_true, y_score),    p_curve, r_curve, _ = precision_recall_curve(y_true, y_score),    auc_pr = auc(r_curve, p_curve),    y_norm = [(s-min(y_score))/(max(y_score)-min(y_score)) for s in y_score],    y_pred = [1 if s >= 0.5 else 0 for s in y_norm],    acc = accuracy_score(y_true, y_pred),    prec = precision_score(y_true, y_pred, zero_division=0),    rec = recall_score(y_true, y_pred, zero_division=0),    f1 = f1_score(y_true, y_pred, zero_division=0),else:,    auc_roc = auc_pr = acc = prec = rec = f1 = 0,,print(f'\n  P@1: {p_at_1:.4f}, P@3: {p_at_3:.4f}, P@5: {p_at_5:.4f}'),print(f'  NDCG@5: {ndcg_5:.4f}'),print(f'  AUC-ROC: {auc_roc:.4f}, AUC-PR: {auc_pr:.4f}'),print(f'  Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}')

## 📈 Step 10: Visualize Results

In [ ]:
print('[8/8] Creating visualizations...'),,fig, axes = plt.subplots(2, 2, figsize=(14, 10)),,# Precision,ax = axes[0, 0],x = np.arange(3),bars1 = ax.bar(x - 0.2, [p_at_1, p_at_3, p_at_5], 0.4, label='Actual', color='steelblue'),bars2 = ax.bar(x + 0.2, [0.45, 0.40, 0.32], 0.4, label='Target', color='coral'),ax.set_xticks(x),ax.set_xticklabels(['P@1', 'P@3', 'P@5']),ax.set_title('Precision@K', fontweight='bold'),ax.legend(),ax.grid(True, alpha=0.3, axis='y'),for bars in [bars1, bars2]:,    for bar in bars:,        h = bar.get_height(),        ax.text(bar.get_x()+bar.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=9),,# AUC,ax = axes[0, 1],x = np.arange(2),bars1 = ax.bar(x - 0.2, [auc_roc, auc_pr], 0.4, label='Actual', color='green'),bars2 = ax.bar(x + 0.2, [0.94, 0.80], 0.4, label='Target', color='coral'),ax.set_xticks(x),ax.set_xticklabels(['AUC-ROC', 'AUC-PR']),ax.set_title('AUC Metrics', fontweight='bold'),ax.legend(),ax.set_ylim(0, 1.1),ax.grid(True, alpha=0.3, axis='y'),for bars in [bars1, bars2]:,    for bar in bars:,        h = bar.get_height(),        ax.text(bar.get_x()+bar.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=9),,# Classification,ax = axes[1, 0],x = np.arange(4),bars = ax.bar(x, [acc, prec, rec, f1], 0.6, color='purple', alpha=0.8),ax.set_xticks(x),ax.set_xticklabels(['Acc', 'Prec', 'Rec', 'F1']),ax.set_title('Classification Metrics', fontweight='bold'),ax.set_ylim(0, 1.1),ax.grid(True, alpha=0.3, axis='y'),for bar in bars:,    h = bar.get_height(),    ax.text(bar.get_x()+bar.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=9),,# NDCG,ax = axes[1, 1],bars1 = ax.bar([0], [ndcg_5], 0.5, label='Actual', color='orange'),bars2 = ax.bar([1], [0.35], 0.5, label='Target', color='coral'),ax.set_xticks([0, 1]),ax.set_xticklabels(['NDCG@5\nActual', 'NDCG@5\nTarget']),ax.set_title('NDCG@5 Ranking', fontweight='bold'),ax.legend(),ax.grid(True, alpha=0.3, axis='y'),for bar in list(bars1) + list(bars2):,    h = bar.get_height(),    ax.text(bar.get_x()+bar.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=9),,plt.tight_layout(),plt.savefig('evaluation_metrics_gpu.png', dpi=300, bbox_inches='tight'),plt.show(),,# Precision distribution,fig, ax = plt.subplots(1, 1, figsize=(10, 6)),ax.hist(p1, bins=20, alpha=0.5, label='P@1', color='steelblue'),ax.hist(p3, bins=20, alpha=0.5, label='P@3', color='green'),ax.hist(p5, bins=20, alpha=0.5, label='P@5', color='coral'),ax.set_xlabel('Precision Score'),ax.set_ylabel('Number of Users'),ax.set_title('Precision Distribution (GPU)', fontweight='bold'),ax.legend(),ax.grid(True, alpha=0.3),plt.tight_layout(),plt.savefig('precision_dist_gpu.png', dpi=300, bbox_inches='tight'),plt.show(),,print('✅ Visualizations created!')

## 📄 Step 11: Summary Report

In [ ]:
print('='*60),print('EVALUATION SUMMARY (GPU-ACCELERATED)'),print('='*60),print(f"\n📊 P@1: {p_at_1:.4f} {'✓' if p_at_1>=0.45 else '✗'} (target: 0.45)"),print(f"    P@3: {p_at_3:.4f} {'✓' if p_at_3>=0.40 else '✗'} (target: 0.40)"),print(f"    P@5: {p_at_5:.4f} {'✓' if p_at_5>=0.32 else '✗'} (target: 0.32)"),print(f"    NDCG@5: {ndcg_5:.4f} {'✓' if ndcg_5>=0.35 else '✗'} (target: 0.35)"),print(f"\n📈 AUC-ROC: {auc_roc:.4f} {'✓' if auc_roc>=0.94 else '✗'} (target: 0.94)"),print(f"    AUC-PR: {auc_pr:.4f} {'✓' if auc_pr>=0.80 else '✗'} (target: 0.80)"),print(f"\n🎯 Acc: {acc:.4f}"),print(f"    Prec: {prec:.4f} {'✓' if prec>=0.12 else '✗'} (target: 0.12)"),print(f"    Rec: {rec:.4f} {'✓' if rec>=0.99 else '✗'} (target: 0.99)"),print(f"    F1: {f1:.4f} {'✓' if f1>=0.22 else '✗'} (target: 0.22)"),,targets_met = sum([p_at_1>=0.45, p_at_3>=0.40, p_at_5>=0.32, auc_roc>=0.94, auc_pr>=0.80, prec>=0.12, rec>=0.99, f1>=0.22, ndcg_5>=0.35]),print(f"\n🎯 OVERALL: {targets_met}/9 targets met ({targets_met/9*100:.1f}%)"),print('='*60),,summary_df = pd.DataFrame({,    'Metric': ['P@1', 'P@3', 'P@5', 'NDCG@5', 'AUC-ROC', 'AUC-PR', 'Acc', 'Prec', 'Rec', 'F1'],,    'Value': [p_at_1, p_at_3, p_at_5, ndcg_5, auc_roc, auc_pr, acc, prec, rec, f1],,    'Target': [0.45, 0.40, 0.32, 0.35, 0.94, 0.80, None, 0.12, 0.99, 0.22],,    'Pass': ['✓' if p_at_1>=0.45 else '✗', '✓' if p_at_3>=0.40 else '✗', '✓' if p_at_5>=0.32 else '✗',,             '✓' if ndcg_5>=0.35 else '✗', '✓' if auc_roc>=0.94 else '✗', '✓' if auc_pr>=0.80 else '✗',,             '—', '✓' if prec>=0.12 else '✗', '✓' if rec>=0.99 else '✗', '✓' if f1>=0.22 else '✗'],}),display(summary_df),summary_df.to_csv('evaluation_summary_gpu.csv', index=False),print('\n✅ Saved: evaluation_summary_gpu.csv')

## 📥 Step 12: Download Results

In [ ]:
files.download('evaluation_metrics_gpu.png'),files.download('precision_dist_gpu.png'),files.download('evaluation_summary_gpu.csv'),print('✅ Download complete!')

## 💾 Step 12: Save Results to Google Drive


In [ ]:
# Save results back to Google Drive
output_path = '/content/drive/MyDrive/data_573/results/'
import os
import shutil

os.makedirs(output_path, exist_ok=True)

shutil.copy('evaluation_metrics_gpu.png', output_path + 'evaluation_metrics_gpu.png')
shutil.copy('precision_dist_gpu.png', output_path + 'precision_dist_gpu.png')
shutil.copy('evaluation_summary_gpu.csv', output_path + 'evaluation_summary_gpu.csv')

print('✅ Results saved to Google Drive!')
print(f'   Location: {output_path}')
print('\n📁 Files saved:')
print('   • evaluation_metrics_gpu.png')
print('   • precision_dist_gpu.png')
print('   • evaluation_summary_gpu.csv')


## 📥 Step 13: (Optional) Download Results Locally


In [ ]:
# Uncomment these lines if you want to download files to your computer
# from google.colab import files
# files.download('evaluation_metrics_gpu.png')
# files.download('precision_dist_gpu.png')
# files.download('evaluation_summary_gpu.csv')
# print('✅ Download complete!')

print('\n✅ EVALUATION COMPLETE!')
print('Check your Google Drive for results!')
